In [1]:
from pyspark.sql import SparkSession

# Crear sesión Spark con conector Kafka
spark = SparkSession.builder \
    .appName("KafkaStreamExample") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0"
    ) \
    .getOrCreate()

In [2]:
dfStream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "proyect-final-topic") \
    .option("startingOffsets", "latest") \
    .load()

In [3]:
df1 = dfStream.select(dfStream["value"].cast("string"))
df1.printSchema()

root
 |-- value: string (nullable = true)



In [4]:
from pyspark.sql.types import StructType
from pyspark.sql.types import StructField
from pyspark.sql.types import StringType



In [5]:
schema = StructType([
    StructField("message", StringType(), True),
    StructField("iss_position", StructType([
        StructField("longitude", StringType(), True),
        StructField("latitude", StringType(), True),
    ]), True),
    StructField("timestamp", StringType(), True)
])

In [6]:
import pyspark.sql.functions as f
df2 = df1.withColumn(
    "json", # nombre de la nueva columna
    f.from_json(df1["value"], schema) #función que convierte el campo value a un json
)
df2.printSchema()

root
 |-- value: string (nullable = true)
 |-- json: struct (nullable = true)
 |    |-- message: string (nullable = true)
 |    |-- iss_position: struct (nullable = true)
 |    |    |-- longitude: string (nullable = true)
 |    |    |-- latitude: string (nullable = true)
 |    |-- timestamp: string (nullable = true)



In [7]:
df3 = df2.select(
    df2["json.message"].cast("string"),
    df2["json.iss_position.longitude"].cast("double"),
    df2["json.iss_position.latitude"].cast("double"),
    df2["json.timestamp"].cast("int")
)
df3.printSchema()

root
 |-- message: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [8]:
dfResultado = df3.filter(df3["message"] == "success")

In [ ]:
# Carpeta dentro de work
output_path = "work/jupyter_storage/resultado_kafka"
checkpoint_path = "work/jupyter_storage/resultado_kafka/_checkpoints/resultado"

# Guardar el stream en formato parquet
dfResultado.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="1 second") \
    .start(output_path) \
    .awaitTermination()